## 第 1 節｜郵件分類、混淆矩陣與資料洩漏

### 資料集欄位｜500 封模擬校園郵件

| 欄位 | 用途 |
|---|---|
| `email_id` | 郵件編號，不拿來訓練 |
| `link_count` | 網址連結數量，輸入 X |
| `urgent_words` | 緊急用語次數，輸入 X |
| `exclamation_count` | 驚嘆號次數，輸入 X |
| `sender_known` | 是否為已知寄件者（1 是、0 否），輸入 X |
| `email_type` | 郵件類型，透過 One-Hot 變成數值欄位 |
| `is_spam` | 是否為垃圾郵件（1 是、0 否），答案 y |

先將 `02_校園郵件分類_500筆.csv` 放在 Notebook 的工作目錄。第一個 Code Cell **直接執行**。


In [ ]:
# Import pandas to read the email dataset.
# 匯入 pandas，讀取校園郵件資料。
import pandas as pd

# Split data into training and test sets.
# 切分訓練集與測試集。
from sklearn.model_selection import train_test_split

# Build the decision tree used in the first lesson.
# 匯入第一節使用的決策樹模型。
from sklearn.tree import DecisionTreeClassifier

# Evaluate classification results and draw the confusion matrix.
# 計算分類指標並繪製混淆矩陣。
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

# Display the confusion matrix graph.
# 顯示混淆矩陣圖表。
import matplotlib.pyplot as plt

# Load the 500 simulated campus emails.
# 載入 500 封模擬校園郵件。
df = pd.read_csv('02_校園郵件分類_500筆.csv')

# Preview the first five records.
# 預覽前五筆郵件資料。
display(df.head())

### 練習 1｜準備 X、y 與測試集（照範例輸入下一格）

```python
# Select the five features used by the model.
# 選取模型需要的五個特徵欄位。
features = ['link_count', 'urgent_words', 'exclamation_count', 'sender_known', 'email_type']

# Apply One-Hot Encoding only to the email type.
# 只對郵件類型進行 One-Hot Encoding，其餘數值欄位保持不變。
X = pd.get_dummies(df[features], columns=['email_type'], dtype=int)

# Select the target labels.
# 選取模型要預測的答案。
y = df['is_spam']

# Reserve 20% of the emails as unseen test data.
# 留下兩成郵件，最後才用來測試模型。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Check the number of emails in each group.
# 確認訓練與測試資料的筆數。
print('訓練:', len(X_train), '測試:', len(X_test))
```

In [ ]:
# 在這格輸入上方「練習 1」的程式碼。


### 練習 2｜訓練決策樹並測試（照範例輸入下一格）

```python
# Create a simple decision tree.
# 建立最多三層的決策樹。
model = DecisionTreeClassifier(max_depth=3, random_state=42)

# Learn from the training examples.
# 讓模型從訓練資料學習。
model.fit(X_train, y_train)

# Predict the 100 unseen emails.
# 預測一百封測試郵件。
pred = model.predict(X_test)

# Count correct predictions and missed spam.
# 查看正確率與漏掉的垃圾郵件數量。
print('Accuracy:', round(accuracy_score(y_test, pred) * 100), '%')

# Count missed spam emails.
# 計算漏抓的垃圾郵件。
print('漏掉垃圾郵件:', int(((y_test == 1) & (pred == 0)).sum()), '封')
```

In [ ]:
# 在這格輸入上方「練習 2」的程式碼。


### Optional 1｜混淆矩陣、Precision、Recall、F1（直接執行）

這一格對應投影片第 3 頁，顯示四種判斷結果與各種評估指標。

In [ ]:
# Calculate how many predicted spam emails were really spam.
# 計算預測為垃圾郵件的信件中，有多少比例真的屬於垃圾郵件。
print('Precision:', round(precision_score(y_test, pred) * 100, 1), '%')

# Calculate how many real spam emails the model caught.
# 計算真正的垃圾郵件中，模型成功抓到多少比例。
print('Recall:', round(recall_score(y_test, pred) * 100, 1), '%')

# Combine precision and recall into one score.
# 使用 F1 綜合衡量 Precision 與 Recall。
print('F1:', round(f1_score(y_test, pred) * 100, 1), '%')

# Show the actual and predicted class counts.
# 顯示正常郵件與垃圾郵件的實際、預測四格結果。
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Normal (0)', 'Spam (1)'], cmap='Blues')

# Display the confusion matrix.
# 顯示混淆矩陣圖表。
plt.show()

### Optional 2｜資料洩漏：錯把答案當特徵（直接執行）

這一格對應投影片第 5 頁。**錯誤做法只用於理解資料洩漏，不能作為正式模型。**

In [ ]:
# Intentionally include the known answer as a feature for demonstration.
# 刻意把正確答案加進輸入特徵，示範錯誤做法。
wrong_features = features + ['is_spam']

# Convert only email_type while keeping the leaked answer column.
# 只對郵件類型進行 One-Hot，並保留錯誤加入的答案欄位。
X_wrong = pd.get_dummies(df[wrong_features], columns=['email_type'], dtype=int)

# Keep the same train/test split as the correct model.
# 使用與正確模型相同的訓練／測試切分。
X_wrong_train, X_wrong_test = train_test_split(X_wrong, test_size=0.2, stratify=y, random_state=42)

# Train a tree that can see the answer in its inputs.
# 訓練一棵能直接看到答案欄位的錯誤模型。
wrong_model = DecisionTreeClassifier(max_depth=3, random_state=42)

# Fit the intentionally incorrect model.
# 訓練這個刻意設計的錯誤模型。
wrong_model.fit(X_wrong_train, y_train)

# Compare valid test accuracy with the misleading leaked score.
# 比較正常測試成績與偷看答案後的虛假高分。
print('正確做法:', round(accuracy_score(y_test, pred) * 100), '%')

# Show the misleading score caused by label leakage.
# 顯示答案外洩所造成的不可信成績。
print('偷放答案:', round(wrong_model.score(X_wrong_test, y_test) * 100), '%')